In [ ]:
EXPERIMENT = 1
DESCRIPTION = "<TEXT DESCRIPTION OF THE EXPERIMENT>"

DATASET_NAME = "tr_uni_exam_questions"
DATASET_VERSION = "v1"
MODEL = "gemma3:12b"
PROMPT_NAME = "tr_uni_exam_system"
PROMPT_VERSION = "v1"
HUMAN_QUERY_NAME = "tr_uni_exam_human_query"
HUMAN_QUERY_VERSION = "v1"
MODEL_PARAMS_NAME = "tr_uni_exam_params"
MODEL_PARAMS_VERSION = "v1"
SCHEMA_NAME = "tr_uni_exam_answer"
SCHEMA_VERSION = "v1"

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from funcs import load_train_data

data, DATASET_REVISION = load_train_data(name=DATASET_NAME, version=DATASET_VERSION)

print(f"Loaded {len(data)} training samples.\n")
print(f"Field names: {list(data[0].keys())}\n")

In [ ]:
import pandas as pd

df = pd.DataFrame(data)

df.head()

In [ ]:
from funcs import load_prompt

SYSTEM_PROMPT, PROMPT_HASH = load_prompt(name=PROMPT_NAME, version=PROMPT_VERSION)

print(f"System prompt:\n{SYSTEM_PROMPT}\n")

In [ ]:
from funcs import load_human_query

build_human_query, HUMAN_QUERY_HASH = load_human_query(name=HUMAN_QUERY_NAME, version=HUMAN_QUERY_VERSION)

print(f"Human query example:\n{build_human_query(data[0])}\n")

In [ ]:
from funcs import load_params

MODEL_PARAMS, MODEL_PARAMS_HASH = load_params(name=MODEL_PARAMS_NAME, version=MODEL_PARAMS_VERSION)
print(f"Model parameters:\n{MODEL_PARAMS}\n")

In [ ]:
from funcs import load_schema

Answer, SCHEMA_HASH = load_schema(name=SCHEMA_NAME, version=SCHEMA_VERSION)
print(f"Schema:\n{Answer.model_json_schema()}\n")

In [ ]:
from funcs import load_human_query

build_human_query, HUMAN_QUERY_HASH = load_human_query(name=HUMAN_QUERY_NAME, version=HUMAN_QUERY_VERSION)

In [ ]:
from funcs import build_manifest, get_model_digest

manifest = build_manifest(
    model=MODEL, 
    model_digest=get_model_digest(MODEL),
    dataset=(DATASET_NAME, DATASET_VERSION, DATASET_REVISION),
    prompt=(PROMPT_NAME, PROMPT_VERSION, PROMPT_HASH), 
    human_query=(HUMAN_QUERY_NAME, HUMAN_QUERY_VERSION, HUMAN_QUERY_HASH),
    params=(MODEL_PARAMS_NAME, MODEL_PARAMS_VERSION, MODEL_PARAMS_HASH), 
    schema=(SCHEMA_NAME, SCHEMA_VERSION, SCHEMA_HASH),
    experiment=EXPERIMENT, description=DESCRIPTION
)
print(f"Manifest:\n{manifest}\n")

In [ ]:
from langchain_ollama import ChatOllama

chat_model = ChatOllama(model=MODEL, **MODEL_PARAMS)

llm = chat_model.with_structured_output(Answer)

In [ ]:
from funcs import ask_to_llm

question = data[0]

answer = ask_to_llm(llm, SYSTEM_PROMPT, build_human_query(question), question["image"])

print(f"LLM answer: {answer}")
print(f"Actual answer: {question['answer']}")

In [ ]:
from funcs import run_experiment

summary = run_experiment(manifest, llm, SYSTEM_PROMPT, build_human_query, data)